# Compare Attention Features from Different Extraction Methods

This notebook compares features extracted by:
1. `experiments/head_detection.py --save_features` (uses core_detector.py, uncalibrated)
2. `scripts/extract_head_features.py` (uses HFFeatureExtractor, calibrated by default)

Run these commands first to generate the files:
```bash
# From experiments/ directory:
cd experiments
python head_detection.py --llm mistral --detector core --temp 0.001 --save_features --max_samples 10

# From scripts/ directory (uncalibrated to match core_detector):
cd scripts
python extract_head_features.py --llm mistral --input_file ../head_data/nq_core.json --max_samples 10 --no_calibration -o compare_test_uncalib

# Also with calibration (default behavior):
python extract_head_features.py --llm mistral --input_file ../head_data/nq_core.json --max_samples 10 -o compare_test_calib
```

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

def mtime(p):
    return datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S") if p.exists() else "N/A"

# Base directory
base_dir = Path('..') / 'head_data' / 'mistral'

## Load Feature Files

In [10]:
# File paths - adjust these based on your actual file names
core_detector_file = base_dir / 'core_temp0.001_prune0.0_features.npz'
hf_uncalib_file    = base_dir / 'compare_test_uncalib.npz'
hf_calib_file      = base_dir / 'compare_test_calib.npz'

# Check which files exist
print("File existence check:")
print(f"  core_detector features: {core_detector_file.exists()} - {core_detector_file}, modified {mtime(core_detector_file)}")
print(f"  HF uncalibrated: {hf_uncalib_file.exists()} - {hf_uncalib_file}, modified {mtime(hf_uncalib_file)}")
print(f"  HF calibrated: {hf_calib_file.exists()} - {hf_calib_file}, modified {mtime(hf_calib_file)}")

File existence check:
  core_detector features: True - ../head_data/mistral/core_temp0.001_prune0.0_features.npz, modified 2026-02-11 13:58:26
  HF uncalibrated: True - ../head_data/mistral/compare_test_uncalib.npz, modified 2026-02-12 11:15:02
  HF calibrated: True - ../head_data/mistral/compare_test_calib.npz, modified 2026-02-12 11:15:39


In [11]:
# Load available files
data = {}

if core_detector_file.exists():
    data['core_detector'] = dict(np.load(core_detector_file, allow_pickle=True))
    print(f"Loaded core_detector: {data['core_detector']['features'].shape}")

if hf_uncalib_file.exists():
    data['hf_uncalib'] = dict(np.load(hf_uncalib_file, allow_pickle=True))
    print(f"Loaded hf_uncalib: {data['hf_uncalib']['features'].shape}")

if hf_calib_file.exists():
    data['hf_calib'] = dict(np.load(hf_calib_file, allow_pickle=True))
    print(f"Loaded hf_calib: {data['hf_calib']['features'].shape}")

Loaded core_detector: (250, 1024)
Loaded hf_uncalib: (250, 1024)
Loaded hf_calib: (250, 1024)


## Compare Shapes and Basic Statistics

In [12]:
print("\n" + "="*70)
print("Feature File Comparison")
print("="*70)

for name, d in data.items():
    features = d['features']
    labels = d['labels']
    docs_per_query = d.get('docs_per_query', None)
    
    print(f"\n{name}:")
    print(f"  Features shape: {features.shape}")
    print(f"  Labels shape: {labels.shape}")
    print(f"  Docs per query: {docs_per_query.shape if docs_per_query is not None else 'N/A'}")
    print(f"  Label distribution: pos={np.sum(labels==1)}, neg={np.sum(labels==0)}, other={np.sum(labels==-1)}")
    print(f"  Feature stats: min={features.min():.6f}, max={features.max():.6f}, mean={features.mean():.6f}, std={features.std():.6f}")


Feature File Comparison

core_detector:
  Features shape: (250, 1024)
  Labels shape: (250,)
  Docs per query: (5,)
  Label distribution: pos=5, neg=245, other=0
  Feature stats: min=0.000002, max=0.176758, mean=0.003497, std=0.006008

hf_uncalib:
  Features shape: (250, 1024)
  Labels shape: (250,)
  Docs per query: (5,)
  Label distribution: pos=5, neg=245, other=0
  Feature stats: min=0.000002, max=0.176758, mean=0.003497, std=0.006008

hf_calib:
  Features shape: (250, 1024)
  Labels shape: (250,)
  Docs per query: (5,)
  Label distribution: pos=5, neg=245, other=0
  Feature stats: min=-0.396400, max=0.139343, mean=-0.004559, std=0.014266


## Compare Feature Values (if shapes match)

In [13]:
def compare_features(f1, f2, name1, name2):
    """Compare two feature arrays."""
    print(f"\nComparing {name1} vs {name2}:")
    print(f"  Shape match: {f1.shape == f2.shape}")
    
    if f1.shape != f2.shape:
        print(f"  Cannot compare: shapes differ ({f1.shape} vs {f2.shape})")
        return None
    
    # Element-wise differences
    diff = f1 - f2
    abs_diff = np.abs(diff)
    
    print(f"  Difference stats:")
    print(f"    Min diff: {diff.min():.8f}")
    print(f"    Max diff: {diff.max():.8f}")
    print(f"    Mean diff: {diff.mean():.8f}")
    print(f"    Std diff: {diff.std():.8f}")
    print(f"    Mean abs diff: {abs_diff.mean():.8f}")
    print(f"    Max abs diff: {abs_diff.max():.8f}")
    
    # Correlation
    corr = np.corrcoef(f1.flatten(), f2.flatten())[0, 1]
    print(f"  Correlation: {corr:.6f}")
    
    # Are they approximately equal?
    close_1e3 = np.allclose(f1, f2, rtol=1e-3, atol=1e-3)
    close_1e5 = np.allclose(f1, f2, rtol=1e-5, atol=1e-5)
    print(f"  Approx equal (rtol=1e-3): {close_1e3}")
    print(f"  Approx equal (rtol=1e-5): {close_1e5}")
    
    return diff

In [14]:
# Compare core_detector vs HF uncalibrated (should be similar if both uncalibrated)
if 'core_detector' in data and 'hf_uncalib' in data:
    diff_uncalib = compare_features(
        data['core_detector']['features'], 
        data['hf_uncalib']['features'],
        'core_detector', 'hf_uncalib'
    )


Comparing core_detector vs hf_uncalib:
  Shape match: True
  Difference stats:
    Min diff: 0.00000000
    Max diff: 0.00000000
    Mean diff: 0.00000000
    Std diff: 0.00000000
    Mean abs diff: 0.00000000
    Max abs diff: 0.00000000
  Correlation: 1.000000
  Approx equal (rtol=1e-3): True
  Approx equal (rtol=1e-5): True


In [15]:
# Compare core_detector vs HF calibrated (should differ due to calibration)
if 'core_detector' in data and 'hf_calib' in data:
    diff_calib = compare_features(
        data['core_detector']['features'], 
        data['hf_calib']['features'],
        'core_detector', 'hf_calib'
    )


Comparing core_detector vs hf_calib:
  Shape match: True
  Difference stats:
    Min diff: 0.00000000
    Max diff: 0.40332031
    Mean diff: 0.00805522
    Std diff: 0.01481034
    Mean abs diff: 0.00805522
    Max abs diff: 0.40332031
  Correlation: 0.118198
  Approx equal (rtol=1e-3): False
  Approx equal (rtol=1e-5): False


In [ ]:
# Compare HF uncalibrated vs calibrated (shows effect of calibration)
if 'hf_uncalib' in data and 'hf_calib' in data:
    diff_calib_effect = compare_features(
        data['hf_uncalib']['features'], 
        data['hf_calib']['features'],
        'hf_uncalib', 'hf_calib'
    )

## Visualize Differences

In [ ]:
def plot_feature_comparison(f1, f2, name1, name2, sample_idx=0):
    """Plot feature comparison for a single sample."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Sample features
    s1 = f1[sample_idx]
    s2 = f2[sample_idx]
    
    # Scatter plot
    ax = axes[0, 0]
    ax.scatter(s1, s2, alpha=0.5, s=10)
    min_val = min(s1.min(), s2.min())
    max_val = max(s1.max(), s2.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', label='y=x')
    ax.set_xlabel(name1)
    ax.set_ylabel(name2)
    ax.set_title(f'Feature Values (sample {sample_idx})')
    ax.legend()
    
    # Difference histogram
    ax = axes[0, 1]
    diff = s1 - s2
    ax.hist(diff, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='r', linestyle='--')
    ax.set_xlabel('Difference')
    ax.set_ylabel('Count')
    ax.set_title(f'Difference Distribution (sample {sample_idx})')
    
    # Feature values by head index
    ax = axes[1, 0]
    ax.plot(s1, label=name1, alpha=0.7)
    ax.plot(s2, label=name2, alpha=0.7)
    ax.set_xlabel('Head Index')
    ax.set_ylabel('Feature Value')
    ax.set_title(f'Feature Values by Head (sample {sample_idx})')
    ax.legend()
    
    # Difference by head index
    ax = axes[1, 1]
    ax.plot(diff)
    ax.axhline(0, color='r', linestyle='--')
    ax.set_xlabel('Head Index')
    ax.set_ylabel('Difference')
    ax.set_title(f'Difference by Head (sample {sample_idx})')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot comparison for first sample
if 'core_detector' in data and 'hf_uncalib' in data:
    if data['core_detector']['features'].shape == data['hf_uncalib']['features'].shape:
        print("Comparing core_detector vs hf_uncalib (both uncalibrated):")
        plot_feature_comparison(
            data['core_detector']['features'],
            data['hf_uncalib']['features'],
            'core_detector', 'hf_uncalib',
            sample_idx=0
        )

In [ ]:
# Plot calibration effect
if 'hf_uncalib' in data and 'hf_calib' in data:
    if data['hf_uncalib']['features'].shape == data['hf_calib']['features'].shape:
        print("Effect of calibration (hf_uncalib vs hf_calib):")
        plot_feature_comparison(
            data['hf_uncalib']['features'],
            data['hf_calib']['features'],
            'hf_uncalib', 'hf_calib',
            sample_idx=0
        )

## Compare Per-Head Statistics Across All Samples

In [ ]:
def plot_head_statistics(data_dict):
    """Plot per-head mean and std across all samples."""
    fig, axes = plt.subplots(len(data_dict), 2, figsize=(14, 4*len(data_dict)))
    if len(data_dict) == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (name, d) in enumerate(data_dict.items()):
        features = d['features']
        
        # Mean per head
        head_means = features.mean(axis=0)
        head_stds = features.std(axis=0)
        
        ax = axes[idx, 0]
        ax.bar(range(len(head_means)), head_means, alpha=0.7)
        ax.set_xlabel('Head Index')
        ax.set_ylabel('Mean Feature Value')
        ax.set_title(f'{name}: Mean per Head')
        
        ax = axes[idx, 1]
        ax.bar(range(len(head_stds)), head_stds, alpha=0.7)
        ax.set_xlabel('Head Index')
        ax.set_ylabel('Std Feature Value')
        ax.set_title(f'{name}: Std per Head')
    
    plt.tight_layout()
    plt.show()

if len(data) > 0:
    plot_head_statistics(data)

In [ ]:
import json

# Load original data
with open('../head_data/nq_core.json', 'r') as f:
    nq_data = json.load(f)

# Check first few documents for whitespace differences
print("Checking whitespace normalization differences:\n")
n_different = 0
for q_idx, query in enumerate(nq_data[:3]):
    for p_idx, para in enumerate(query['paragraphs'][:3]):
        text = para['paragraph_text']
        
        # Method 1: head_detection.py style
        method1 = text.strip()
        
        # Method 2: extract_head_features.py style  
        method2 = ' '.join(text.split())
        
        if method1 != method2:
            n_different += 1
            print(f"Query {q_idx}, Para {p_idx}: DIFFERENT")
            print(f"  Original length: {len(text)}")
            print(f"  After strip(): {len(method1)}")
            print(f"  After split/join: {len(method2)}")
            print(f"  Diff: {len(method1) - len(method2)} chars")
            # Show where they differ
            for i, (c1, c2) in enumerate(zip(method1, method2)):
                if c1 != c2:
                    print(f"  First diff at char {i}: '{repr(c1)}' vs '{repr(c2)}'")
                    print(f"  Context: ...{repr(method1[max(0,i-10):i+10])}...")
                    break
            print()

print(f"\nTotal documents with whitespace differences: {n_different}")

## Fix: Normalize whitespace in head_detection.py

To make them match, change `head_detection.py` line 75 from:
```python
documents = [(p['paragraph_text']).strip() for p in paragraphs]
```
to:
```python
documents = [' '.join(p['paragraph_text'].split()) for p in paragraphs]
```

Or change `extract_head_features.py` to use `.strip()` instead of split/join.

## Check Whitespace Normalization Difference

The key difference is how documents are preprocessed:
- `head_detection.py`: `doc.strip()` - only removes leading/trailing whitespace
- `extract_head_features.py`: `' '.join(text.split())` - normalizes ALL whitespace

## Compare Labels

In [ ]:
print("\n" + "="*70)
print("Label Comparison")
print("="*70)

for name, d in data.items():
    labels = d['labels']
    print(f"\n{name}:")
    print(f"  Labels: {labels[:20]}...")
    print(f"  Unique values: {np.unique(labels)}")

# Check if labels match between files
names = list(data.keys())
if len(names) >= 2:
    l1 = data[names[0]]['labels']
    l2 = data[names[1]]['labels']
    if len(l1) == len(l2):
        print(f"\nLabels match between {names[0]} and {names[1]}: {np.array_equal(l1, l2)}")

## Detailed Sample-by-Sample Comparison

In [ ]:
def compare_sample_by_sample(f1, f2, name1, name2, num_samples=5):
    """Compare features sample by sample."""
    if f1.shape != f2.shape:
        print("Cannot compare: shapes differ")
        return
    
    print(f"\nSample-by-sample comparison ({name1} vs {name2}):")
    print(f"{'Sample':<8} {'Corr':<10} {'MAE':<12} {'MaxDiff':<12} {'Match?':<8}")
    print("-" * 55)
    
    for i in range(min(num_samples, len(f1))):
        s1 = f1[i]
        s2 = f2[i]
        
        corr = np.corrcoef(s1, s2)[0, 1]
        mae = np.abs(s1 - s2).mean()
        max_diff = np.abs(s1 - s2).max()
        match = np.allclose(s1, s2, rtol=1e-3, atol=1e-3)
        
        print(f"{i:<8} {corr:<10.6f} {mae:<12.8f} {max_diff:<12.8f} {str(match):<8}")

In [ ]:
if 'core_detector' in data and 'hf_uncalib' in data:
    compare_sample_by_sample(
        data['core_detector']['features'],
        data['hf_uncalib']['features'],
        'core_detector', 'hf_uncalib',
        num_samples=10
    )

## Summary

In [ ]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print("""
Key differences to check:

1. CALIBRATION:
   - core_detector.py: NO calibration (raw attention)
   - extract_head_features.py: Calibration by default (subtract N/A query attention)
   - Use --no_calibration flag to match core_detector behavior

2. DOCUMENT ORDER:
   - Both should process documents in the same order
   - Check that labels match between files

3. NUMERICAL PRECISION:
   - Small differences (~1e-6) are expected due to floating point
   - Larger differences indicate implementation differences

If core_detector and hf_uncalib features don't match closely:
   - Check tokenization differences (offset handling)
   - Check document truncation (max_doc_tokens)
   - Check attention weight computation (_get_attn_weights)
""")